In [5]:
# ШАГ 1 — Проверка null значений в исходных данных
import requests

# Читаем исходные данные
df_raw = spark.read.csv("/home/jovyan/work/restaurant_csv/", header=True, inferSchema=True)
print("Всего ресторанов:", df_raw.count())

# Ищем null координаты
nulls = df_raw.filter(col("lat").isNull() | col("lng").isNull())
print("Ресторанов с null координатами:", nulls.count())
nulls.show()

# ШАГ 2 — Получаем координаты через OpenCage API
API_KEY = "59707b01457a4ac3b1e2ee35922be98c"

def get_coordinates(city, country):
    url = "https://api.opencagedata.com/geocode/v1/json"
    params = {"q": f"{city}, {country}", "key": API_KEY, "limit": 1}
    data = requests.get(url, params=params).json()
    if data["results"]:
        return data["results"][0]["geometry"]["lat"], data["results"][0]["geometry"]["lng"]
    return None, None

lat, lng = get_coordinates("Dillon", "US")
print(f"Координаты Dillon, US: lat={lat}, lng={lng}")

Всего ресторанов: 1997
Ресторанов с null координатами: 1
+-----------+------------+--------------+-----------------------+-------+------+----+----+
|         id|franchise_id|franchise_name|restaurant_franchise_id|country|  city| lat| lng|
+-----------+------------+--------------+-----------------------+-------+------+----+----+
|85899345920|           1|       Savoria|                  18952|     US|Dillon|NULL|NULL|
+-----------+------------+--------------+-----------------------+-------+------+----+----+

Координаты Dillon, US: lat=34.4014089, lng=-79.3864339


In [1]:
import subprocess
subprocess.run(["pip", "install", "pygeohash"], capture_output=True)

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, lit, udf, avg
from pyspark.sql.types import StringType
import pygeohash as gh

spark = SparkSession.builder \
    .appName("SparkPractice") \
    .config("spark.driver.memory", "6g") \
    .config("spark.sql.shuffle.partitions", "20") \
    .getOrCreate()

def encode_geohash(lat, lng):
    if lat is None or lng is None:
        return None
    return gh.encode(lat, lng, precision=4)

geohash_udf = udf(encode_geohash, StringType())

# Рестораны
df = spark.read.csv("/home/jovyan/work/restaurant_csv/", header=True, inferSchema=True)
df_fixed = df.withColumn("lat", when(col("lat").isNull(), lit(34.4014089)).otherwise(col("lat"))) \
             .withColumn("lng", when(col("lng").isNull(), lit(-79.3864339)).otherwise(col("lng")))
df_with_geohash = df_fixed.withColumn("geohash", geohash_udf(col("lat"), col("lng")))
df_with_geohash.cache()

restaurant_geohashes = [row.geohash for row in df_with_geohash.select("geohash").distinct().collect()]
print(f"Spark готов! Geohash ресторанов: {len(restaurant_geohashes)}")

# Обрабатываем по одной папке
for i in range(1, 9):
    print(f"Обрабатываем weather_{i}...")
    path = f"/home/jovyan/work/weathers/weather_{i}/"
    
    df_temp = spark.read.option("basePath", path).parquet(path)
    df_temp = df_temp.withColumn("geohash", geohash_udf(col("lat"), col("lng")))
    df_temp = df_temp.filter(col("geohash").isin(restaurant_geohashes))
    
    df_agg = df_temp.groupBy("geohash", "wthr_date") \
        .agg(avg("avg_tmpr_c").alias("avg_tmpr_c"),
             avg("avg_tmpr_f").alias("avg_tmpr_f"))
    
    result = df_with_geohash.join(df_agg, on="geohash", how="left")
    
    result.write \
        .mode("append") \
        .partitionBy("country") \
        .parquet("/home/jovyan/work/output/")
    
    spark.catalog.clearCache()
    df_with_geohash.cache()
    print(f"weather_{i} сохранён!")

print("Всё готово! 🎉")

Spark готов! Geohash ресторанов: 692
Обрабатываем weather_1...
weather_1 сохранён!
Обрабатываем weather_2...
weather_2 сохранён!
Обрабатываем weather_3...
weather_3 сохранён!
Обрабатываем weather_4...
weather_4 сохранён!
Обрабатываем weather_5...
weather_5 сохранён!
Обрабатываем weather_6...
weather_6 сохранён!
Обрабатываем weather_7...
weather_7 сохранён!
Обрабатываем weather_8...
weather_8 сохранён!
Всё готово! 🎉


In [2]:
result_check = spark.read.parquet("/home/jovyan/work/output/")
print("Строк в результате:", result_check.count())
print("Колонки:", result_check.columns)
result_check.show(5)

Строк в результате: 66697
Колонки: ['geohash', 'id', 'franchise_id', 'franchise_name', 'restaurant_franchise_id', 'city', 'lat', 'lng', 'wthr_date', 'avg_tmpr_c', 'avg_tmpr_f', 'country']
+-------+------------+------------+------------------+-----------------------+--------+------+--------+----------+------------------+-----------------+-------+
|geohash|          id|franchise_id|    franchise_name|restaurant_franchise_id|    city|   lat|     lng| wthr_date|        avg_tmpr_c|       avg_tmpr_f|country|
+-------+------------+------------+------------------+-----------------------+--------+------+--------+----------+------------------+-----------------+-------+
|   9t9p|111669149700|           5|  The Hungry Hippo|                  84488|  Tucson| 32.22|-110.877|2016-10-08|            25.632|78.13600000000001|     US|
|   9t9p|111669149700|           5|  The Hungry Hippo|                  84488|  Tucson| 32.22|-110.877|2016-10-07|            24.868|76.77599999999998|     US|
|   9t9p|111

In [4]:
# ===== ТЕСТЫ исправленные =====
print("Запускаем тесты...\n")

# Тест 1 — нет null координат
nulls_count = df_with_geohash.filter(col("lat").isNull() | col("lng").isNull()).count()
assert nulls_count == 0
print("✅ Тест 1 пройден: null координат нет")

# Тест 2 — geohash ровно 4 символа
from pyspark.sql.functions import length
wrong_geohash = df_with_geohash.filter(length(col("geohash")) != 4).count()
assert wrong_geohash == 0
print("✅ Тест 2 пройден: все geohash длиной 4 символа")

# Тест 3 — результат не пустой
result_count = result_check.count()
assert result_count > 0
print(f"✅ Тест 3 пройден: в результате {result_count} строк")

# Тест 4 — количество ресторанов не изменилось
restaurant_count = df_with_geohash.select("id").distinct().count()
result_restaurant_count = result_check.select("id").distinct().count()
assert restaurant_count == result_restaurant_count
print(f"✅ Тест 4 пройден: все {restaurant_count} ресторанов в результате")

# Тест 5 — файл сохранён в parquet
import os
assert os.path.exists("/home/jovyan/work/output/")
print("✅ Тест 5 пройден: output папка существует")

print("\n🎉 Все тесты пройдены!")

Запускаем тесты...

✅ Тест 1 пройден: null координат нет
✅ Тест 2 пройден: все geohash длиной 4 символа
✅ Тест 3 пройден: в результате 66697 строк
✅ Тест 4 пройден: все 1997 ресторанов в результате
✅ Тест 5 пройден: output папка существует

🎉 Все тесты пройдены!
